# ValueLens Refresh Model (Databricks)

Databricks adaptation of the corresponding ValueLens notebook. It uses the configured Unity Catalog schema and volume; see the [Databricks route guide](../README.md) for setup and scheduling.


In [ ]:
from valuelens_databricks_runtime import initialize
VL = initialize(spark, dbutils)
DB_SCHEMA = VL.schema
VL.record_task_start()


In [ ]:
WORKSPACE_ID = VL.widget("pbi_workspace_id", "")
SEMANTIC_MODEL_ID = VL.widget("pbi_semantic_model_id", "")
WRITE_MODE = 'merge'     # 'overwrite' after a backfill reloads the whole audit table
TIMEOUT_MINUTES = 120    # Give up waiting for a refresh after this long
CHECK_AUDIT_STATE = True  # verify the curated audit table changed before refreshing

TENANT_ID = VL.widget("tenant_id", "")
CLIENT_ID = VL.widget("client_id", "")
CLIENT_SECRET = VL.secret()


In [ ]:
import re
import time

import requests

AUDIT_TABLE = 'Chat + Agent Interactions (Audit Logs)'
BUSY_RETRY_MINUTES = 30
FINAL = {'Completed', 'Failed', 'Cancelled', 'Disabled', 'TimedOut'}

if not WORKSPACE_ID or not SEMANTIC_MODEL_ID:
    raise ValueError('Set Power BI workspace and semantic model IDs in the cell above.')
GUID = r"^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$"
if not re.fullmatch(GUID, WORKSPACE_ID) or not re.fullmatch(GUID, SEMANTIC_MODEL_ID):
    raise ValueError("Power BI workspace and semantic model IDs must be GUIDs.")

REFRESHES = f'https://api.powerbi.com/v1.0/myorg/groups/{WORKSPACE_ID}/datasets/{SEMANTIC_MODEL_ID}/refreshes'



AUDIT_SOURCE = "copilot_interactions_curated"
AUDIT_DATA_VERSION_PROPERTY = "valuelens.audit_data_version"


def _audit_refresh_state():
    if not CHECK_AUDIT_STATE:
        return ""
    # Fail clearly if the curated Delta table is unavailable.
    detail = spark.sql(f"DESCRIBE DETAIL {AUDIT_SOURCE}").collect()[0]
    properties = detail["properties"] or {}
    pending = properties.get(AUDIT_DATA_VERSION_PROPERTY, "")
    completed = properties.get(f"valuelens.audit_refreshed.{SEMANTIC_MODEL_ID}", "")
    return pending if pending != completed else ""


def _acknowledge_audit_refresh(token):
    if not token:
        return
    # A separate acknowledgement cannot erase a newer processor token during a concurrent run.
    key = f"valuelens.audit_refreshed.{SEMANTIC_MODEL_ID}".replace("\\", "\\\\").replace("'", "\\'")
    value = token.replace("\\", "\\\\").replace("'", "\\'")
    spark.sql(f"ALTER TABLE {AUDIT_SOURCE} SET TBLPROPERTIES ('{key}' = '{value}')")


def _headers():
    # A fresh token for each call: a long refresh can outlive one.
    return VL.powerbi_headers(TENANT_ID, CLIENT_ID, CLIENT_SECRET)


def _history(top=10):
    r = requests.get(REFRESHES, headers=_headers(), params={'$top': top}, timeout=60)
    r.raise_for_status()
    return r.json().get('value', [])


def _start(body):
    """Starts a refresh and returns its ID. Waits while another refresh is running."""
    deadline = time.time() + BUSY_RETRY_MINUTES * 60
    while True:
        r = requests.post(REFRESHES, headers=_headers(), json=body, timeout=60)
        if r.status_code == 202:
            location = r.headers.get('Location', '')
            return location.rstrip('/').split('/')[-1] or r.headers.get('RequestId')
        busy = r.status_code in (409, 429) or 'in progress' in r.text.lower()
        if busy and time.time() < deadline:
            print(f'Another refresh is running ({r.status_code}). Trying again in a minute.')
            time.sleep(60)
            continue
        raise RuntimeError(f'Power BI would not start the refresh ({r.status_code}): {r.text[:1000]}')


def _wait(refresh_id):
    deadline = time.time() + TIMEOUT_MINUTES * 60
    started = time.time()
    while True:
        r = requests.get(f'{REFRESHES}/{refresh_id}', headers=_headers(), timeout=60)
        if r.status_code == 200:
            d = r.json()
            status = d.get('extendedStatus') or d.get('status')
            if status in FINAL:
                # A refresh can end Completed with a Warning or Error message, e.g. a calculated table
                # whose DAX does not parse. Its tables, columns and relationships are then never
                # calculated and report pages fail, so treat it as a failed refresh.
                problems = [m for m in d.get('messages', []) if m.get('type') in ('Warning', 'Error')]
                if status != 'Completed' or problems:
                    messages = '; '.join(str(m.get('message', m)) for m in (problems or d.get('messages', [])))
                    ending = status + (' with warnings' if status == 'Completed' else '')
                    raise RuntimeError(f'The refresh ended as {ending}. {messages}'.strip())
                print(f'  done in {round((time.time() - started) / 60, 1)} min')
                return d
        elif r.status_code not in (404, 429, 500, 502, 503, 504):
            r.raise_for_status()
        if time.time() > deadline:
            raise TimeoutError(f'The refresh was still running after {TIMEOUT_MINUTES} minutes.')
        time.sleep(30)


# Capture the committed generation before attempting SQL endpoint synchronization.
audit_refresh_token = _audit_refresh_state()
print('Databricks SQL Warehouse reads the configured Unity Catalog schema directly.')

first_refresh = not any(h.get('status') == 'Completed' for h in _history())
print('Refreshing the semantic model' + (' for the first time' if first_refresh else '') + '...')
_wait(_start({'type': 'full', 'commitMode': 'transactional', 'applyRefreshPolicy': True, 'retryCount': 1}))

if audit_refresh_token or (WRITE_MODE == 'overwrite' and not first_refresh):
    # An incremental refresh only reloads recent days. The curated table was rebuilt, so reload all of it.
    print(f'Reloading every partition of "{AUDIT_TABLE}" after historical data changed...')
    _wait(_start({
        'type': 'full',
        'commitMode': 'transactional',
        'applyRefreshPolicy': False,
        'retryCount': 1,
        'objects': [{'table': AUDIT_TABLE}],
    }))

_acknowledge_audit_refresh(audit_refresh_token)

print('The semantic model is up to date.')

VL.record_task_success()
